# Un siècle de PIB américain : deux courbes, deux histoires · *A century of U.S. GDP: two curves, two stories*

Notebook compagnon du chapitre **10. PIB réel et PIB nominal : pourquoi corriger de l'inflation** — [lire l'article](https://nmlab.io/ressources/pib-reel-pib-nominal).
Companion notebook to chapter **10. Real GDP and Nominal GDP: Why Correct for Inflation** — [read the article](https://nmlab.io/en/ressources/real-gdp-nominal-gdp).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_gdp() -> tuple[Series, Series]:
    """PIB américain nominal (GDPA, dollars courants) et réel (GDPCA, dollars chaînés de 2017),
    séries annuelles depuis 1929, en direct depuis FRED.
    U.S. nominal (GDPA) and real (GDPCA) GDP, annual since 1929, live from FRED."""
    return nm.load_fred("GDPA"), nm.load_fred("GDPCA")

nominal, real = load_gdp()


import pandas as pd
import matplotlib.dates as mdates
from matplotlib.ticker import NullFormatter
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Un siècle de PIB américain : deux courbes, deux histoires",
        sub="Avant l'année de base, le réel est au-dessus ; après, en dessous — l'écart, ce sont les prix",
        ylab="milliards de dollars, échelle log",
        legend_nom="PIB nominal (dollars courants)",
        legend_real="PIB réel (dollars chaînés de 2017)",
        base="2017 : année de base\nnominal = réel",
        start_nom="1929 : 105 Mds courants",
        start_real="aux prix de 2017 : 1 191 Mds",
        note="En dollars courants, le PIB de 1929 vaut 105 milliards ; aux prix de 2017, 1 191 milliards.\n"
             "La pente bleue mélange volumes et prix ; la rouge ne garde que les volumes. Source : BEA via FRED."),
    "en": dict(
        title="A century of U.S. GDP: two curves, two stories",
        sub="Before the base year, real sits above; after, below — the gap is prices",
        ylab="billions of dollars, log scale",
        legend_nom="nominal GDP (current dollars)",
        legend_real="real GDP (chained 2017 dollars)",
        base="2017: base year\nnominal = real",
        start_nom="1929: $105bn current",
        start_real="at 2017 prices: $1,191bn",
        note="In current dollars, 1929 GDP is worth 105 billion; at 2017 prices, 1,191 billion.\n"
             "The blue slope mixes volumes and prices; the red keeps only volumes. Source: BEA via FRED."),
}

def _at(series: Series, year: int):
    """Renvoie (date, valeur) du premier point de l'année ``year``."""
    mask = series.index.year == year
    return series.index[mask][0], float(series[mask].iloc[0])

def build_figure(nominal: Series, real: Series, lang: str) -> Figure:
    """Deux courbes en échelle log, 1929-2025, avec l'année de base 2017 et les points extrêmes."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1140)
    ax = nm.axes(fig, left=0.088)
    ax.plot(nominal.index, nominal, color=nm.COLORS["blue"], lw=3.2, label=text["legend_nom"], zorder=3)
    ax.plot(real.index, real, color=nm.COLORS["rose"], lw=3.2, label=text["legend_real"], zorder=3)
    ax.set_yscale("log")
    ax.set_ylim(48, 42000)
    ax.set_yticks([100, 1000, 10000, 30000])
    ax.yaxis.set_major_formatter(nm.thousands(lang))
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_ylabel(text["ylab"])
    ax.set_xlim(pd.Timestamp("1926-01-01"), pd.Timestamp("2033-01-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator(20))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    d29_dip, v29_dip = _at(nominal, 1933)
    d29_real, v29_real = _at(real, 1929)
    ax.annotate(text["start_nom"], xy=(d29_dip, v29_dip), xytext=(90, 34), textcoords="offset points",
                ha="left", va="center", fontsize=21, color=nm.COLORS["blue"],
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))
    ax.annotate(text["start_real"], xy=(d29_real, v29_real), xytext=(88, -18), textcoords="offset points",
                ha="left", va="center", fontsize=21, color=nm.COLORS["rose"],
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))

    d17, v17 = _at(nominal, 2017)
    ax.scatter([d17], [v17], s=150, color=nm.COLORS["text"], zorder=6)
    ax.annotate(text["base"], xy=(d17, v17), xytext=(-46, 92), textcoords="offset points",
                ha="center", va="center", fontsize=22, color=nm.COLORS["text"], linespacing=1.5,
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["muted"], lw=1.5))

    fmt = nm.thousands(lang)
    ax.annotate(fmt(nominal.iloc[-1], None), xy=(nominal.index[-1], nominal.iloc[-1]),
                xytext=(10, 4), textcoords="offset points", ha="left", va="center",
                fontsize=25, fontweight="bold", color=nm.COLORS["blue"])
    ax.annotate(fmt(real.iloc[-1], None), xy=(real.index[-1], real.iloc[-1]),
                xytext=(10, -2), textcoords="offset points", ha="left", va="center",
                fontsize=25, fontweight="bold", color=nm.COLORS["rose"])

    ax.legend(loc="lower right", frameon=False, fontsize=21, labelcolor=nm.COLORS["text"],
              handlelength=1.6, borderaxespad=1.2)
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(nominal, real, LANG)